In [ ]:
# Practice Problems Day 12

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 12 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---


**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 6 assumes you have done them. Plan on about 45 minutes.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron chapter 11, **Batch Normalization**, **Faster Optimizers** and **Learning Rate Scheduling**.

GOALS:

1. Switch a network from plain SGD to Adam.
2. Add a learning rate schedule to a training run.
3. Explain training mode and prediction mode in words.

**How much of this should you write yourself?** The starting cell is given, it is the network from the Day 11 practice. Problem 1 changes one line. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 12 practice"
git push
```

---

In [ ]:
# RUN THIS. The California housing data from the notes, split, scaled, and turned into tensors.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

housing = pd.read_csv("data/california_housing.csv")
X = housing.drop(columns=["MedHouseVal"])
y = housing["MedHouseVal"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_valid_t = torch.tensor(y_valid.values, dtype=torch.float32).reshape(-1, 1)
train_data = TensorDataset(X_train_t, y_train_t)               # pairs the features with the labels for the DataLoader

print("training rows:", X_train_t.shape[0], "   validation rows:", X_valid_t.shape[0])

In [ ]:
# RUN THIS. The network from the Day 11 practice, Problem 2a: 4 hidden layers, Kaiming, plain SGD.
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)                # the optimizer

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step, once per BATCH

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

## Problem 1: Switch to Adam

**What to copy, what to write:** Copy the cell above into the cell below. Change only the optimizer line.

**1a.** Change the optimizer to **Adam** with a learning rate of 0.001 and run it. How does it compare to plain SGD after 1 epoch and after 20 epochs?

**1b.** In your own words, why can Adam do better than plain SGD with the same learning rate? Use the word "adaptive".

In [ ]:
# 1a: the same cell with Adam
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)                # Adam, the usual starting learning rate

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step, once per BATCH

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

**1a.** Plain SGD was at 0.8788 after 1 epoch and 0.6388 after 20. Adam was at 0.7043 after 1 epoch and 0.5538 after 20. Adam is better the whole way, and after just 5 epochs (0.5876) it is already better than SGD after 20!

**1b.** "Adaptive" means every weight gets its own step size. Adam keeps track of how big each weight's gradients have been lately and divides the step by that size. Weights with tiny gradients still get a decent sized step, and weights with huge gradients do not jump too far. Plain SGD uses the same learning rate for every weight, so one learning rate has to work for all of them.

---

## Problem 2: Add a learning rate schedule (steps outlined)

**What to copy, what to write:** Copy your Adam cell from Problem 1 and add the schedule yourself. It is the same pattern as the "real life" cell at the end of the notes.

**2a.** Add a `StepLR` schedule with `step_size=10` and `gamma=0.1` to your Adam cell, and train for 20 epochs.

**2b.** Compare the validation RMSE at epochs 15 and 20, with and without the schedule. Did the schedule help here?

**2c.** Where does `scheduler.step()` go in the loop, and where does `optimizer.step()` go? Why are they in different places?

In [ ]:
# 2a: Adam with a StepLR schedule
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)                # Adam
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,                                # make the schedule AFTER the optimizer
    step_size=10,                             # every 10 epochs...
    gamma=0.1)                                # ...multiply the learning rate by 0.1

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step, once per BATCH
    scheduler.step()                          # lower the learning rate on schedule, once per EPOCH

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

**2a and 2b.**

| | epoch 15 | epoch 20 |
|---|---|---|
| Adam | 0.5553 | 0.5538 |
| Adam with the schedule | 0.5404 | 0.5402 |

The first 10 epochs are exactly the same, because the learning rate is the same. After epoch 10 the schedule drops the learning rate to 0.0001. Here the schedule helped a little: 0.5402 against 0.5538, and it barely moved from epoch 15 to 20 because the steps are small now.

**2c.** `optimizer.step()` goes inside the batch loop, so it happens once per **batch**: it updates the weights. `scheduler.step()` goes after the batch loop, so it happens once per **epoch**: it changes the learning rate. If you put `scheduler.step()` inside the batch loop, the learning rate would drop every 10 batches instead of every 10 epochs, and it would be tiny almost right away.

---

## Problem 3: In words

**3a.** A classmate added batch normalization to their network. Their validation RMSE comes out a little different every time they change the batch size they use for the validation rows, even though the model has not changed. What did they forget, and why does it cause this?

**3b.** A friend's network with 10 hidden layers gives a loss of `nan` when they train it with plain SGD. Name three things from Day 11 and Day 12 they could try, in the order you would try them, and say why each one could help.

**3a.** They forgot `model.eval()` before predicting. In training mode a batch normalization layer standardizes with the mean and standard deviation of whatever batch it is given, so each row's prediction depends on the other rows in its batch. Change the batches and the predictions change. In prediction mode it uses the running averages saved during training, and the batch no longer matters.

**3b.** Your order can be different, here is mine:

1. **A smaller learning rate.** `nan` means the steps are too big, and this is the fastest thing to try.
2. **ReLU and Kaiming initialization.** They keep the signal and the gradients a good size through all 10 layers.
3. **Adam**, which gives every weight its own step size. Batch normalization is also a fair answer, it keeps the signal a good size during training.

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** Go back to the starting cell and change the optimizer to SGD with `momentum=0.9` and a learning rate of **0.0001**. Compare it to plain SGD with a learning rate of 0.001. Why are they so close?

**B.** Train your Adam cell with a learning rate of 0.01 instead of 0.001. Is the bigger learning rate better here?

In [ ]:
# Optional A: SGD with momentum and a 10 times smaller learning rate
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)                # momentum

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step, once per BATCH

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

In [ ]:
# Optional B: Adam with a learning rate of 0.01
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)                # a 10 times bigger learning rate

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step, once per BATCH

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

**A.** Momentum with a learning rate of 0.0001 ends at 0.6391, almost exactly plain SGD with 0.001 (0.6388). Momentum makes the steps up to ten times bigger, and the learning rate is ten times smaller, so they take about the same size steps.

**B.** With a learning rate of 0.01 Adam ends at 0.5467, a little better than 0.5538 here, but look at epoch 10 (0.6163): bigger steps bounce around more.